Note: this notebook compresses an image `cat.jpeg`. I did not attach this image to the repository so users should replace that with their own image.

### Image Compression

There are a couple ways to perform singular value decomposition with code. Considering SVD is so important in the age of data science, the easiest way is just to use some built in library. However, it is best to understand how SVD is calculated.

Consider a matrix $A\in\mathbb{R}^{n\times m}$. Let $A$ be a grayscale image, meaning at each index, it has value between $0$ and $255$, with $0$ being black and $255$ being white.

It is possible to say, using SVD, $A = U\Sigma V^\top$. As you may recall, $U$ is the matrix of left singular vectors with size $n\times n$, $\Sigma$ is the matrix of singular values, padded with zeros to the dimension $n \times m$, and $V$ is the matrix of right singular vectors with size $m \times m$.

Using matrix multiplication, we notice that the dimensions of $A$ remain accurate! $(n \times n) \times (n \times m) \times (m \times m) = (n \times m)$.

Because $\Sigma$ is basically a diagonal matrix, we can further split up $A$ into
$$
A = \sum_{i = 1}^{\min(n, m)} \mathbf{u}_i \sigma_i \mathbf{v}_i^\top
$$
Note: we tend to zero index in code.

Here is how we really "compress" this image: because we sort $\sigma$ such that $\sigma_1 \geq \dots \geq \sigma_{\min(n, m)}$, the first element of our decomposition ($\mathbf{u}_1\sigma_1\mathbf{v}_1^\top$) is the most important element - it captures the most variation. Meanwhile, the $\min(n, m)$th element is the least important, it doesn't keep as much variation.

When reducing to the $k$th dimension, we take the first $k$ terms of this decomposition.

Now you know the math - let's implement!

#### Implementation

Start with importing necessary packages.

In [ ]:
!pip install numpy matplotlib pillow

In [ ]:
import numpy as np # we will be using this to do our calculations
import matplotlib.pyplot as plt # to show the image
from PIL import Image # to mess around with the image

We want to load the image as gray scale so we don't need to tinker with a bunch of channels.

In [ ]:
img = Image.open("cat.jpeg")

plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()

img = img.convert("L")

plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()

Next, we convert this image into a numpy array.

In [ ]:
A = np.array(img, dtype=float)

print(A.shape)

Decompose matrix A into the three other matrices: $U\Sigma V^\top$.

A natural question is why not find the eigenvalues for both $A^\top A$ and $AA^\top$. This is because it is 1) easier to just solve the equation below rather than computing eigenvalues twice and 2) there may arise sign errors when using that approach, among others.

$$
A\mathbf{v}_i = \sigma_i \mathbf{u}_i
$$

In [ ]:
ATA = A.T @ A

# We use .eigh instead of .eig so we can get sorted values returned
eigenvalues, V = np.linalg.eigh(ATA)

# The values returned are in ascending order, we want descending
V = V[:, ::-1]
eigenvalues = eigenvalues[::-1]

# We can't square root negative numbers so we set a lower bound of 0,
# but no upper bound
singular_values = np.sqrt(np.clip(eigenvalues, 0, None))

# We need to derive U now
# We only get so many singular values due to the size of our matrix
num_values = min(A.shape)

# A = U\Sigma V^T
# AV = U\Sigma
V = V[:, :num_values]
singular_values = singular_values[:num_values]

U = np.zeros((A.shape[0], num_values))

for i in range(num_values):
    sigma = singular_values[i]

    # No division by zero
    if not np.isclose(singular_values[i], 0):
        U[:, i] = (A @ V[:, i]) / sigma

Now we have the right singular vectors, left singular vectors, and singular values - all sorted. Considering we already have the left/right singular vectors, we don't need to separately construct $U$ and $V$. After all, they are just the left/right singular vectors.

We do, however, need to construct $\Sigma$. The singular values reside on diagonals, and everything else is zero.

In [ ]:
# We already shrunk the vectors so we can just make a diagonal matrix
sigma = np.diag(singular_values)

We can now reconstruct A with $U\Sigma V^\top$. Let's check:

In [ ]:
print(np.allclose(A, U @ sigma @ V.T))

Let's say we want to pick out the most important $k$ features, let $k = 5$ for now (we can experiment around later). Then, we want to pick the first $k$ entries.

In [ ]:
k = 5

compressed = np.zeros((A.shape[0], A.shape[1]))
for i in range(k):
    # the outer product is just u_i v_i^T
    compressed = np.add(compressed, sigma[i, i] * np.outer(U[:, i], V[:, i]))

plt.imshow(compressed, cmap="gray")
plt.axis("off")
plt.show()

#### The Easy Way

Instead of calculating $U\Sigma V^\top$ manually, we can just use the built in function `np.linalg.svd`, which is more numerically stable than manual computation:

In [ ]:
U, sigma, VT = np.linalg.svd(A)

In [ ]:
k = 5

compressed = np.zeros((A.shape[0], A.shape[1]))
for i in range(k):
    # V is already transposed!
    # Sigma is now just the singular values
    compressed = np.add(compressed, sigma[i] * np.outer(U[:, i], VT[i, :]))

plt.imshow(compressed, cmap="gray")
plt.axis("off")
plt.show()